# 03 — QAOA single run

One QAOA configuration end-to-end on `mag7` with `K=2, p=3`:
the multi-start training curve, the probability histogram, and the top-5
most likely bitstrings (with their true `C(x)`).

Defaults: 10 random `(gamma, beta)` inits, COBYLA `maxiter=200`,
`rhobeg=0.1`. Single-seed QAOA is not meaningful — the landscape is
non-convex enough that the multi-start is mandatory.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
import numpy as np
import matplotlib.pyplot as plt

from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force
from scripts.qaoa      import solve, decode_top_k, make_hamiltonians
from scripts.metrics   import approximation_ratio, prob_optimal, prob_feasible
from scripts.plotting  import apply_style, PALETTE, title, fig_path
apply_style()

In [ ]:
# === Canonical instance: n=16, K=4, p=3 ===
P = 3

r  = load_universe()
pf = PortfolioProblem(r.mu, r.Sigma,
                      lam=DEFAULTS['lam'], A=DEFAULTS['A'],
                      K=DEFAULTS['K_AT_16'], tickers=r.tickers)

bf  = brute_force(pf)
res = solve(pf, p=P, n_restarts=10, seed=42, verbose=False)

print(f'brute force:  x={bf.bitstring}  C={bf.cost:.6f}')
print(f'              picks: {bf.tickers(pf)}')
print(f'QAOA p={P}:     E={res["energy"]:.6f}  ratio={res["ratio"]:.4f}')
print(f'P(optimum)  = {prob_optimal(res["probs"], bf.x):.4f}')
print(f'P(feasible) = {prob_feasible(res["probs"], pf.n, pf.K):.4f}')

### Top-5 bitstrings

In [ ]:
import pandas as pd
pd.DataFrame(decode_top_k(res['probs'], pf, k=5))[['bitstring', 'prob', 'cost', 'budget']]

### Figures — training restarts + probability histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Multi-seed restart energies — shows the landscape is non-convex
energies = [h['fun'] for h in res['history']]
axes[0].bar(range(len(energies)), energies,
            color=PALETTE['blue_muted'], edgecolor=PALETTE['charcoal'])
axes[0].axhline(res['energy'], color=PALETTE['red'], ls='--', lw=1.5,
                label=f'best E = {res["energy"]:.4f}')
axes[0].axhline(bf.cost, color=PALETTE['ochre'], ls=':', lw=1.5,
                label=f'C(x*) = {bf.cost:.4f}')
axes[0].set_xlabel('restart'); axes[0].set_ylabel('converged energy')
title(axes[0], 'Multi-start COBYLA energies',
      f'{len(energies)} random inits; non-convex landscape -> wide spread')
axes[0].legend()

# Probability over budget-feasible bitstrings only (n=16 -> 65k states is too many to plot)
probs = res['probs']
n_states = len(probs)
feas_mask = np.array([bin(k).count('1') == pf.K for k in range(n_states)])
feas_probs = probs[feas_mask]
order = np.argsort(-feas_probs)
top = min(40, len(feas_probs))
axes[1].bar(range(top), feas_probs[order][:top],
            color=PALETTE['red'], edgecolor=PALETTE['charcoal'], linewidth=0.4)
axes[1].set_xlabel(f'rank (top {top} of {len(feas_probs)} feasible states)')
axes[1].set_ylabel('probability')
title(axes[1], 'QAOA probability mass on feasible states',
      f'budget K={pf.K}; uniform baseline = 1/{n_states:_}')
axes[1].axhline(1.0/n_states, color=PALETTE['grey'], ls=':', lw=1, label='uniform')
axes[1].legend()

plt.tight_layout()
fig.savefig(fig_path('qaoa', f'single_run_universe_p{P}'), bbox_inches='tight')
plt.show()